# CS 131 Project Assignment 3 — PySpark, Parquet, and DuckDB (Team 9)

Dataset: UCI Online Retail II (both workbook sheets combined into one TSV).

Team roles this sprint: Kanishka Yadav (Data Engineer), Sanjini Shukla (Data Engineer, notebook integrator),
Lipi Himanshukumar Makwana (Product Manager), Kushan Mehta (Data Storyteller).

Run from `pa3/notebooks/` with Java 17 or 21. Set `PA3_DATA_PATH` to the combined TSV if it is not in one of the
default locations checked in Section 0.

## 0. Environment setup and versions

Spark runs in local mode (`local[*]`) on one machine, using the class-style session. The session time zone is UTC
so that timestamp parsing does not depend on the machine's daylight-saving rules.

ANSI mode is turned **off** explicitly. Spark 4.x turns it on by default, and in ANSI mode an invalid value
makes `cast` raise an error instead of returning `null`. Section 3a counts cast failures as "value present but
`null` after the cast", which needs the non-ANSI behaviour (the Spark 3.x default) on every machine.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T
from pathlib import Path
import os
import sys
import platform
import pandas as pd
import duckdb

# Spark workers must use the same Python as this kernel (avoids a version-mismatch error on some machines).
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

spark = (
    SparkSession.builder
    .appName("CS131-PA3-Online-Retail")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("DuckDB version:", duckdb.__version__)
print("Python version:", platform.python_version())
print("Spark master:", spark.sparkContext.master)
print("Spark ran locally:", spark.sparkContext.master.startswith("local"))

### 0a. Paths

Run this notebook from `pa3/notebooks/`. Input is chosen in this order:

1. `PA3_DATA_PATH`, if it is set.
2. The full combined TSV in one of the default locations outside the repo.
3. **The committed 1,000-row sample** `pa3/data/sample/online_retail_sample.tsv`, so a teammate or grader can run
   every cell without downloading the full dataset. Counts in the outputs will then be sample counts, and checks
   that only hold for the full data (the exact row count) are skipped.
   A sample run writes to `pa3/out/sample_run/` and `pa3/data/parquet/sample_run/` (both gitignored), so it can
   never overwrite the full-data deliverables in `pa3/out/`.

On IBM:

```bash
export PA3_DATA_PATH=../online_retail_II_combined.tsv
```

In [ ]:
PA3_DIR = Path.cwd().parent
REPO_DIR = PA3_DIR.parent

SAMPLE_PATH = PA3_DIR / "data" / "sample" / "online_retail_sample.tsv"
FULL_ROWS = 1_067_371  # data rows in the full combined TSV (PA2: 1,067,372 lines including the header)

DATA_PATH = os.environ.get("PA3_DATA_PATH")

if DATA_PATH:
    DATA_PATH = Path(DATA_PATH)
else:
    candidates = [
        REPO_DIR.parent / "online_retail_II_combined.tsv",
        REPO_DIR.parent / "data" / "online_retail_II_combined.tsv",
        SAMPLE_PATH,
    ]
    DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is None or not DATA_PATH.exists():
    raise FileNotFoundError(
        "Set PA3_DATA_PATH to the combined Online Retail II TSV file, "
        f"or run from pa3/notebooks/ so the committed sample ({SAMPLE_PATH}) is found."
    )

USING_SAMPLE = DATA_PATH.resolve() == SAMPLE_PATH.resolve()

OUT_DIR = PA3_DIR / "out"
PARQUET_DIR = PA3_DIR / "data" / "parquet" / "online_retail_prepared"
DEV_SAMPLE_DIR = PA3_DIR / "data" / "dev_sample_20pct"

# A run on the committed sample must never overwrite the full-data deliverables or Parquet.
# These folders are all gitignored.
if USING_SAMPLE:
    OUT_DIR = PA3_DIR / "out" / "sample_run"
    PARQUET_DIR = PA3_DIR / "data" / "parquet" / "sample_run" / "online_retail_prepared"
    DEV_SAMPLE_DIR = PA3_DIR / "data" / "parquet" / "sample_run" / "dev_sample_20pct"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", DATA_PATH)
print("Input size (bytes):", f"{DATA_PATH.stat().st_size:,}")
print("Using committed 1,000-row sample:", USING_SAMPLE)
print("Outputs:", OUT_DIR)

## 1. Project goal, questions, and data description

*Owner: Lipi (text) — integrated by Sanjini.* Goal, the two questions, row meaning, key fields, important columns,
data types, and units. The data types must match the schema contract in Section 3b.

## 2. Read the data with PySpark

*Owner: Kanishka.*

- **Input:** the combined Online Retail II TSV (both workbook sheets), 1,067,371 transaction rows plus one header.
- **Explicit schema:** all eight columns are read as `StringType`. Casting happens in Section 3, so that cast
  failures can be counted instead of being silently turned into nulls by the reader.
- **Quoting is turned off** (`quote=""`). About 2,500 descriptions contain literal `"` characters. With quoting off,
  Spark splits every line on tabs exactly as PA2's `cut`/`awk` did, so no row can be merged or shifted.
- **`mode="FAILFAST"`** stops the read if any line does not parse into the eight expected fields.
- An empty field is read as `null` (Spark's default `nullValue` is the empty string).

In [ ]:
RAW_COLUMNS = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]

raw_schema = T.StructType([
    T.StructField(column, T.StringType(), True)
    for column in RAW_COLUMNS
])

raw_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("quote", "")
    .option("mode", "FAILFAST")
    .schema(raw_schema)
    .csv(str(DATA_PATH))
)

raw_rows = raw_df.count()
print("Rows:", raw_rows)
if USING_SAMPLE:
    assert raw_rows == 1_000, f"Expected the committed sample to have 1,000 rows, got {raw_rows:,}"
else:
    assert raw_rows == FULL_ROWS, f"Expected {FULL_ROWS:,} rows (PA2: 1,067,372 lines including the header)"

raw_df.printSchema()
raw_df.show(5, truncate=False)

## 3. Select and cast columns

*Owner: Kanishka.*

Columns are renamed to snake_case and cast to their analysis types. Identifier and text columns are kept
**exactly as read**: no trimming, no upper- or lower-casing. This matters for Analysis 1, because PA2 counted
`StockCode` values case-sensitively (for example `85123A` and `85123a` are different codes) and kept one code
with a trailing space (`"47503J "`).

| Column | Source | Type | Unit / meaning |
|---|---|---|---|
| `invoice` | Invoice | string | Invoice number; a leading `C` marks a cancellation |
| `stock_code` | StockCode | string (unchanged) | Product code |
| `description` | Description | string | Product name |
| `quantity` | Quantity | integer | Units of the product on this invoice line |
| `invoice_date` | InvoiceDate | timestamp (`yyyy-MM-dd HH:mm:ss`) | Invoice date and time |
| `price` | Price | double | Price per unit, pounds sterling (GBP) |
| `customer_id` | Customer ID | string | Customer identifier (an ID, not a measure) |
| `country` | Country | string (unchanged) | Customer's country |

**Row meaning:** one row is one invoice line — one product on one invoice.

In [ ]:
typed_df = raw_df.select(
    F.col("Invoice").alias("invoice"),
    F.col("StockCode").alias("stock_code"),
    F.col("Description").alias("description"),
    F.col("Quantity").cast("int").alias("quantity"),
    F.to_timestamp(
        F.col("InvoiceDate"),
        "yyyy-MM-dd HH:mm:ss"
    ).alias("invoice_date"),
    F.col("Price").cast("double").alias("price"),
    F.col("Customer ID").alias("customer_id"),
    F.col("Country").alias("country"),
)

typed_df.printSchema()
typed_df.show(5, truncate=False)

### 3a. Cast failures and missing key values

A **cast failure** is a row whose raw value is present but becomes `null` after casting. A **blank** value is a
non-null string that is empty after trimming; it is counted here but the stored value is not modified.

**Missing-value rules** (applied the same way in PySpark and DuckDB, because DuckDB reads the same Parquet):

| Case | Rule |
|---|---|
| Null or blank `stock_code` | Expected 0. If any appear, Analysis 1 keeps them as their own group (as PA2's `uniq -c` would have); Analysis 2 excludes them. |
| Null or blank `country` | Expected 0. Same rule as `stock_code`. `Unspecified` is a real value and stays as its own country. |
| Missing or invalid `quantity` / `price` | Expected 0 cast failures. Rows are kept; they get `revenue = null` and are flagged by the quality check in Section 5. |
| Missing or invalid `invoice_date` | Expected 0 cast failures. Rows are kept. |
| Missing `customer_id` | Kept and flagged in Section 5. |

In [ ]:
# Cast failure = raw value present, but null after the same cast used in typed_df.
# Blank = non-null string that is empty after trimming (counted only; values are not modified).
checks = raw_df.agg(
    F.sum(F.when(F.col("Quantity").isNotNull() & F.col("Quantity").cast("int").isNull(), 1).otherwise(0)).alias("quantity_cast_failures"),
    F.sum(F.when(F.col("Price").isNotNull() & F.col("Price").cast("double").isNull(), 1).otherwise(0)).alias("price_cast_failures"),
    F.sum(F.when(F.col("InvoiceDate").isNotNull() & F.to_timestamp(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss").isNull(), 1).otherwise(0)).alias("invoice_date_cast_failures"),
    F.sum(F.when(F.col("Quantity").isNull(), 1).otherwise(0)).alias("null_quantity"),
    F.sum(F.when(F.col("Price").isNull(), 1).otherwise(0)).alias("null_price"),
    F.sum(F.when(F.col("InvoiceDate").isNull(), 1).otherwise(0)).alias("null_invoice_date"),
    F.sum(F.when(F.col("StockCode").isNull(), 1).otherwise(0)).alias("null_stock_code"),
    F.sum(F.when(F.trim(F.col("StockCode")) == "", 1).otherwise(0)).alias("blank_stock_code"),
    F.sum(F.when(F.col("Country").isNull(), 1).otherwise(0)).alias("null_country"),
    F.sum(F.when(F.trim(F.col("Country")) == "", 1).otherwise(0)).alias("blank_country"),
    F.sum(F.when(F.col("Customer ID").isNull() | (F.trim(F.col("Customer ID")) == ""), 1).otherwise(0)).alias("missing_customer_id"),
    F.sum(F.when(F.col("StockCode") != F.trim(F.col("StockCode")), 1).otherwise(0)).alias("stock_code_with_edge_whitespace"),
).first().asDict()

for name, value in checks.items():
    print(f"{name:34s} {value:>10,}")

must_be_zero = [
    "quantity_cast_failures", "price_cast_failures", "invoice_date_cast_failures",
    "null_quantity", "null_price", "null_invoice_date",
    "null_stock_code", "blank_stock_code", "null_country", "blank_country",
]
nonzero = {name: checks[name] for name in must_be_zero if checks[name] != 0}
assert not nonzero, f"Unexpected missing or invalid values: {nonzero}"
print("\nAll cast-failure and missing-key checks are zero.")

### 3b. Schema contract for downstream sections

*Handoff from Kanishka to Sanjini and Kushan.* Sections 4–8 add to this DataFrame; Sections 9–12 read the
**reloaded Parquet** from Section 7, not `typed_df`.

| Column | Type | Added in | Notes |
|---|---|---|---|
| `invoice` | string | 3 | Unchanged |
| `stock_code` | string | 3 | **Unchanged — do not trim or change case** |
| `description` | string | 3 | Literal text (quoting was off, so any `"` characters are kept) |
| `quantity` | int | 3 | Units |
| `invoice_date` | timestamp | 3 | Session time zone is UTC |
| `price` | double | 3 | GBP per unit |
| `customer_id` | string | 3 | `null` when missing |
| `country` | string | 3 | Unchanged |
| `revenue` | double | 4 | `quantity * price`, GBP; `null` if either input is `null`; negative for returns |
| `is_cancellation` | boolean | 4 | `invoice` starts with `C` |
| `flag_missing_customer` | boolean | 5 | `customer_id IS NULL` |
| `flag_bad_qty_price` | boolean | 5 | `quantity` or `price` is `null` or ≤ 0 |

Parquet location (generated locally, gitignored): `pa3/data/parquet/online_retail_prepared/`.

## 4. Derived columns

*Owner: Kanishka.*

| Column | Type | Rule | Why it is useful |
|---|---|---|---|
| `revenue` | double | `quantity * price`, GBP. `null` if either input is `null`. Negative for returns. | Line value; the main measure for spending questions. |
| `is_cancellation` | boolean | `invoice` starts with `C` | Separates cancellations from purchases without dropping them. |

`revenue` is not rounded here. Rounding is done only when a result table is displayed or exported, so it is never
rounded twice.

In [ ]:
derived_df = (
    typed_df
    .withColumn("revenue", F.col("quantity") * F.col("price"))
    .withColumn("is_cancellation", F.col("invoice").startswith("C"))
)

derived_df.select("invoice", "stock_code", "quantity", "price", "revenue", "is_cancellation") \
    .orderBy(F.col("is_cancellation").desc(), "invoice") \
    .show(5, truncate=False)

# Spot check: revenue equals quantity * price on a few rows (Python arithmetic, independent of Spark).
for row in derived_df.select("quantity", "price", "revenue").limit(5).collect():
    assert abs(row["quantity"] * row["price"] - row["revenue"]) < 1e-9
print("revenue spot check passed")

## 5. Data-quality checks

*Owner: Kanishka.*

**Key:** the data has **no natural key**. There is no line-item ID, and `(invoice, stock_code)` is not unique,
because the same product can appear on more than one line of an invoice. The checks below show how many rows
share an `(invoice, stock_code)` pair and how many rows are exact copies of another row.

**The two main checks** (both become flag columns stored in the Parquet):

| Check | Rule | Decision |
|---|---|---|
| A. Missing customer | `customer_id IS NULL` | **Kept** and flagged (`flag_missing_customer`). These are real sales without a customer account. They count toward product and country totals, but customer-level work must filter them out. |
| B. Non-positive or missing quantity / price | `quantity` or `price` is `null` or ≤ 0 | **Kept** and flagged (`flag_bad_qty_price`). Most are cancellations, returns, or manual adjustments. PA1's stock-code frequency counted every row, so Analysis 1 must keep them to reproduce it. PA2's `awk` filter removed exactly these rows; analyses about completed purchases can do the same with `NOT flag_bad_qty_price`. |

**Overlap:** a row can fail both A and B, so the number of rows with at least one problem is
`A + B − (A and B)`, **not** `A + B`. `quality_checks.csv` reports the overlap and the union as separate rows.

**Exact duplicate rows are kept.** Rows that are identical in all eight columns could be true duplicates, but
they could also be the same product scanned twice on one invoice. Without a line-item ID we cannot tell these
apart, so nothing is removed. The duplicates are counted two ways:
- *duplicate groups*: distinct row values that appear more than once (PA2's 32,907 figure counted these);
- *extra copies*: rows that would disappear if each group were reduced to one row.

In [ ]:
flagged_df = (
    derived_df
    .withColumn("flag_missing_customer", F.col("customer_id").isNull())
    .withColumn(
        "flag_bad_qty_price",
        F.col("quantity").isNull() | F.col("price").isNull()
        | (F.col("quantity") <= 0) | (F.col("price") <= 0),
    )
)

total_rows = flagged_df.count()

flag_counts = flagged_df.agg(
    F.sum(F.col("flag_missing_customer").cast("int")).alias("a"),
    F.sum(F.col("flag_bad_qty_price").cast("int")).alias("b"),
    F.sum((F.col("flag_missing_customer") & F.col("flag_bad_qty_price")).cast("int")).alias("a_and_b"),
    F.sum((F.col("flag_missing_customer") | F.col("flag_bad_qty_price")).cast("int")).alias("a_or_b"),
    F.sum(F.col("is_cancellation").cast("int")).alias("cancel"),
    F.sum((F.col("is_cancellation") & F.col("flag_bad_qty_price")).cast("int")).alias("cancel_and_b"),
).first()

# The union must equal A + B - overlap; adding A and B would double count the overlap.
assert flag_counts["a_or_b"] == flag_counts["a"] + flag_counts["b"] - flag_counts["a_and_b"]

# Exact duplicates across all eight original columns.
dup_groups = (
    typed_df.groupBy(typed_df.columns).count()
    .filter(F.col("count") > 1)
    .agg(F.count("*").alias("groups"), F.sum(F.col("count") - 1).alias("extra_copies"))
    .first()
)
dup_group_count = dup_groups["groups"] or 0
dup_extra_copies = dup_groups["extra_copies"] or 0
assert total_rows - typed_df.dropDuplicates().count() == dup_extra_copies

# Rows whose (invoice, stock_code) pair appears on more than one row: evidence that the pair is not a key.
pair_rows = (
    typed_df.groupBy("invoice", "stock_code").count()
    .filter(F.col("count") > 1)
    .agg(F.sum("count").alias("rows"))
    .first()["rows"] or 0
)

for name, value in [
    ("total rows", total_rows),
    ("A. missing customer_id", flag_counts["a"]),
    ("B. quantity/price null or <= 0", flag_counts["b"]),
    ("A and B (overlap)", flag_counts["a_and_b"]),
    ("A or B (union)", flag_counts["a_or_b"]),
    ("cancellations (invoice starts with C)", flag_counts["cancel"]),
    ("cancellations that also fail B", flag_counts["cancel_and_b"]),
    ("exact duplicate groups", dup_group_count),
    ("exact duplicate extra copies", dup_extra_copies),
    ("rows sharing (invoice, stock_code)", pair_rows),
]:
    print(f"{name:40s} {value:>10,}")

### 5a. Save `out/quality_checks.csv`

One row per check. `pct_of_rows` is the share of all prepared rows, rounded to 2 decimals. `overlap_note` explains
how a check relates to the others, so the counts are never simply added.

In [ ]:
def pct(n):
    return round(100.0 * n / total_rows, 2) if total_rows else 0.0


quality_rows = [
    ("Q0", "total_rows", "All rows read from the input", total_rows,
     "Population for every percentage below", ""),
    ("Q1", "cast_failures", "Raw quantity, price, or invoice_date present but null after casting",
     checks["quantity_cast_failures"] + checks["price_cast_failures"] + checks["invoice_date_cast_failures"],
     "Asserted to be 0 (Section 3a)", ""),
    ("Q2", "missing_customer_id", "customer_id is null (empty field in the TSV)", flag_counts["a"],
     "Kept; flag_missing_customer = true", f"{flag_counts['a_and_b']:,} of these rows also fail Q3"),
    ("Q3", "nonpositive_or_missing_qty_price", "quantity or price is null or <= 0", flag_counts["b"],
     "Kept; flag_bad_qty_price = true", f"{flag_counts['a_and_b']:,} of these rows also fail Q2"),
    ("Q4", "overlap_q2_and_q3", "Rows failing both Q2 and Q3", flag_counts["a_and_b"],
     "Informational", "Counted in both Q2 and Q3"),
    ("Q5", "union_q2_or_q3", "Rows failing Q2 or Q3 (at least one)", flag_counts["a_or_b"],
     "Informational", "Q2 + Q3 - Q4; not Q2 + Q3"),
    ("Q6", "cancellation_rows", "invoice starts with 'C'", flag_counts["cancel"],
     "Kept; is_cancellation = true", f"{flag_counts['cancel_and_b']:,} of these rows also fail Q3"),
    ("Q7", "exact_duplicate_extra_copies", "Rows identical in all 8 columns to an earlier row", dup_extra_copies,
     "Kept; cannot prove true duplicates without a line-item ID",
     f"Spread over {dup_group_count:,} duplicate groups (PA2 reported the group count)"),
    ("Q8", "rows_sharing_invoice_stock_code", "(invoice, stock_code) appears on more than one row", pair_rows,
     "Informational; shows (invoice, stock_code) is not a key", "Includes all Q7 rows"),
    ("Q9", "stock_code_edge_whitespace", "stock_code has leading or trailing whitespace",
     checks["stock_code_with_edge_whitespace"], "Kept unchanged to match PA2 grouping", ""),
]

quality_pdf = pd.DataFrame(
    quality_rows,
    columns=["check_id", "check_name", "rule", "rows_affected", "action", "overlap_note"],
)
quality_pdf.insert(4, "pct_of_rows", quality_pdf["rows_affected"].map(pct))

quality_pdf.to_csv(OUT_DIR / "quality_checks.csv", index=False)
print("Wrote", OUT_DIR / "quality_checks.csv")
quality_pdf

## 6. Write Parquet

*Owner: Kanishka.*

The prepared DataFrame (8 cast columns, 2 derived columns, 2 flags) is written as Parquet, replacing any earlier run.
The Parquet folder is generated locally and is gitignored.

In [ ]:
prepared_columns = [
    "invoice", "stock_code", "description", "quantity", "invoice_date", "price",
    "customer_id", "country", "revenue", "is_cancellation",
    "flag_missing_customer", "flag_bad_qty_price",
]
prepared_to_write = flagged_df.select(*prepared_columns)

prepared_to_write.write.mode("overwrite").parquet(str(PARQUET_DIR))

parquet_bytes = sum(p.stat().st_size for p in PARQUET_DIR.rglob("*.parquet"))
print("Wrote", PARQUET_DIR)
print("Parquet size (bytes):", f"{parquet_bytes:,}")

## 7. Read Parquet back and verify

*Owner: Kanishka.*

From here on, every section uses `prepared`, the DataFrame read back from Parquet, not the in-memory `flagged_df`.
The check compares the row count, the column names and order, and each column's data type.

In [ ]:
prepared = spark.read.parquet(str(PARQUET_DIR))

prepared_rows = prepared.count()
expected_types = [(f.name, f.dataType.simpleString()) for f in prepared_to_write.schema.fields]
reloaded_types = [(f.name, f.dataType.simpleString()) for f in prepared.schema.fields]

print("Rows before write:", f"{total_rows:,}")
print("Rows after reload:", f"{prepared_rows:,}")
prepared.printSchema()

assert prepared_rows == total_rows, "Row count changed after the Parquet round trip"
assert reloaded_types == expected_types, f"Schema changed:\n{expected_types}\n{reloaded_types}"
assert prepared.filter(F.col("flag_missing_customer")).count() == flag_counts["a"]
print("Parquet round trip verified: same row count, column order, and data types.")

## 8. 20% development sample and sample comparison

*Owner: Kanishka.*

The development sample uses the method from class on the reloaded Parquet:

```python
sample = prepared.sample(withReplacement=False, fraction=0.20, seed=131)
```

Spark keeps each row independently with probability 0.20, so the sample has *about* 20% of the rows, not exactly
20%. With the same seed, the same input, and the same machine setup, the sample is the same on every run (a fresh
run reproduced `sample_comparison.csv` exactly). Spark samples each partition separately, so a machine with a
different number of cores can split the data differently and select a slightly different 20%. The sample is for fast development
only; Analysis 1, Analysis 2, and the benchmark use the full `prepared` data.

The comparison covers three groupings: **country** (every country, so a missing one is visible),
**is_cancellation**, and **flag_missing_customer**. `pct_point_diff` is sample % minus population %.

In [ ]:
SAMPLE_FRACTION = 0.20
SAMPLE_SEED = 131

sample = prepared.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SAMPLE_SEED)

population_rows = prepared_rows
sample_rows = sample.count()
assert sample_rows > 0, "The development sample is empty"

print("Fraction:", SAMPLE_FRACTION, "| seed:", SAMPLE_SEED)
print("Population rows:", f"{population_rows:,}")
print("Sample rows:    ", f"{sample_rows:,}", f"({100 * sample_rows / population_rows:.2f}% of population)")

sample.write.mode("overwrite").parquet(str(DEV_SAMPLE_DIR))
print("Saved development sample to", DEV_SAMPLE_DIR)

In [ ]:
def compare_group(column):
    pop = prepared.groupBy(column).agg(F.count("*").alias("population_rows"))
    smp = sample.groupBy(column).agg(F.count("*").alias("sample_rows"))
    return (
        pop.join(smp, on=column, how="left")        # left join: keep groups that are missing from the sample
        .fillna(0, subset=["sample_rows"])
        .select(
            F.lit(column).alias("group_column"),
            F.col(column).cast("string").alias("group_value"),
            "population_rows",
            "sample_rows",
        )
    )


comparison_pdf = pd.concat(
    [compare_group(c).toPandas() for c in ["country", "is_cancellation", "flag_missing_customer"]],
    ignore_index=True,
)
comparison_pdf["population_pct"] = (100 * comparison_pdf["population_rows"] / population_rows).round(4)
comparison_pdf["sample_pct"] = (100 * comparison_pdf["sample_rows"] / sample_rows).round(4)
comparison_pdf["pct_point_diff"] = (comparison_pdf["sample_pct"] - comparison_pdf["population_pct"]).round(4)
comparison_pdf["in_sample"] = comparison_pdf["sample_rows"] > 0
comparison_pdf.insert(0, "seed", SAMPLE_SEED)
comparison_pdf.insert(0, "fraction", SAMPLE_FRACTION)

# Deterministic order: grouping, then largest population group first, then name.
comparison_pdf = comparison_pdf.sort_values(
    ["group_column", "population_rows", "group_value"], ascending=[True, False, True]
).reset_index(drop=True)

# Each grouping must cover the whole population and the whole sample.
for column, part in comparison_pdf.groupby("group_column"):
    assert part["population_rows"].sum() == population_rows, column
    assert part["sample_rows"].sum() == sample_rows, column

comparison_pdf.to_csv(OUT_DIR / "sample_comparison.csv", index=False)
print("Wrote", OUT_DIR / "sample_comparison.csv")

missing = comparison_pdf.loc[~comparison_pdf["in_sample"], ["group_column", "group_value", "population_rows"]]
if missing.empty:
    print("Every group in the population also appears in the sample.")
else:
    print("Groups present in the population but missing from the sample:")
    print(missing.to_string(index=False))

print("Largest absolute difference (percentage points):", comparison_pdf["pct_point_diff"].abs().max())
comparison_pdf[comparison_pdf["group_column"] != "country"]

## 9. Analysis 1 — stock-code frequency (PySpark)

*Owner: Sanjini. To be completed.*

## 10. Analysis 2 — top three stock codes per country (window)

*Owner: Sanjini. To be completed.*

## 11. Analysis 1 in DuckDB and row-by-row comparison

*Owner: Sanjini. To be completed.*

## 12. Benchmark

*Owner: Sanjini. To be completed.*

## 13. Chart

*Owner: Kushan (code) — integrated by Sanjini. To be completed.*

## 14. Findings and limitations

*Owner: Kushan (text) — integrated by Sanjini. To be completed.*

## 15. AI-use disclosure

*Owner: Lipi (collects from all members) — integrated by Sanjini. To be completed.*